# День 4 · тот же агент, но инструмент — по MCP

Разница с `agent.py` только в том, откуда берётся инструмент памяти: там — прямой импорт функции
Python, здесь — отдельный процесс (`mcp_memory_server.py`), с которым мы говорим по протоколу
**MCP** через stdin/stdout. Это ровно то, что произойдёт с любым настоящим внешним MCP-сервером:
граф, узлы, `interrupt`, checkpointer — всё то же самое, инструмент просто пришёл по проводу, а не
импортом.

In [ ]:
"""Тот же агент, что в agent.py, но инструмент памяти приходит по MCP, а не импортом Python-функции —
так же, как будет с настоящим внешним MCP-сервером."""
import asyncio
import os
import sys
import threading

import labkit  # noqa: F401  подключает .env до запуска дочернего процесса
from langchain_mcp_adapters.client import MultiServerMCPClient
from agent import build_graph, pricing, run
from tools import search_docs

# --- НАСТРОЙКИ ---
QUESTION = "Найди в памяти заметку про Ollama в проекте ai-labs"
# labkit.ROOT, не Path(__file__): этот файл — ноутбук, а в ячейке Jupyter __file__ не определён.
MEMORY_SERVER = labkit.ROOT / "day4-agent" / "mcp_memory_server.py"


def run_async(coro):
    """asyncio.run(), но безопасно и в обычном скрипте, и в Jupyter — там уже крутится свой event
    loop, и обычный asyncio.run() внутри него падает с RuntimeError. Если цикл уже запущен —
    досчитываем корутину в отдельном потоке с собственным свежим циклом; исключение из потока
    пробрасываем в вызывающий поток сами — по умолчанию поток его просто печатает и теряет."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)                  # обычный скрипт: активного цикла нет, путь как всегда
    box: dict = {}

    def worker():
        try:
            box["value"] = asyncio.run(coro)
        except BaseException as exc:              # ловим и исключения-не-Exception (например, SystemExit)
            box["error"] = exc

    thread = threading.Thread(target=worker)
    thread.start()
    thread.join()
    if "error" in box:
        raise box["error"]
    return box["value"]

## MCP-клиент: спросить у сервера, какие у него есть инструменты

`MultiServerMCPClient` запускает `mcp_memory_server.py` как дочерний процесс и говорит с ним по
`stdio` — тому самому транспорту MCP, которым Claude Code подключается к серверам инструментов.
`client.get_tools()` возвращает их в виде обычных инструментов LangChain — дальше граф не отличает
их от `search_docs`, импортированного напрямую.

In [ ]:
async def main(question: str):
    client = MultiServerMCPClient({"memory": {
        "command": sys.executable,                                              # тот же python, что и здесь
        "args": [str(MEMORY_SERVER)],
        "transport": "stdio",
        # Выделенный клиент знает write-policy до получения инструментов.
        "env": {**os.environ, "MCP_MEMORY_WRITES": "1"},
    }})
    tools = await client.get_tools()             # спрашивает у MCP-сервера список инструментов
    print("MCP tools:", [t.name for t in tools])
    graph = build_graph([search_docs, *tools], await pricing())
    await run(graph, question, thread_id="mcp")

In [ ]:
if __name__ == "__main__":
    run_async(main(QUESTION))